# Regime backtesting baseline

Load and resample bars, project regimes to execution time, scan Keltner variants and model fees, slippage and stops. Some trailing sections are placeholders in the original.

**Status:** curated research source; original saved outputs preserved. Original notes about results are historical observations, not revalidated conclusions. Read `docs/DATA_AND_RUNNING.md` before execution.

Source: `baseline/2026-05-10 Clean Code.ipynb`, original cells 0–62 (zero-based).


In [ ]:
from pathlib import Path
import os
import sys

# Locate the repository from its root or any notebook subdirectory.
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "tradinguni_paths.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the TradingUni repository.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from tradinguni_paths import data_path, output_path
os.chdir(REPO_ROOT)


# Part 1: Data loading + resample + feature/regime

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## Config

In [2]:
DATA_DIR = Path(str(data_path("0.Data -> Facts/Binance Vision Data/aggBars")))
SYMBOL = "ETH"
YEARS = range(2020, 2026)

BASE_COLS = ["t", "open", "high", "low", "close", "vol_base"]
TIMEFRAMES = {
    "15m": "15min",
    "1h": "1h",
    "4h": "4h",
    "1d": "1D",
}

## Load and clean raw 1m data

In [3]:
def load_aggbar_years(
    data_dir: Path,
    symbol: str,
    years,
    cols=BASE_COLS,
) -> pd.DataFrame:
    """
    Load yearly 1m aggBars and return one clean, sorted DataFrame.

    Convention:
    - t is treated as bar-open time.
    - OHLCV columns describe the candle starting at t.
    """
    frames = []

    for year in years:
        path = data_dir / f"aggBars-1m-{symbol}-{year}.parquet"
        d = pd.read_parquet(path, columns=cols)
        frames.append(d)

    df = pd.concat(frames, ignore_index=True)
    df = clean_ohlcv(df, cols=cols)
    return df

In [4]:
def clean_ohlcv(df: pd.DataFrame, cols=BASE_COLS) -> pd.DataFrame:
    """Basic OHLCV hygiene before any resampling/backtest work."""
    d = df.loc[:, cols].copy()

    d["t"] = pd.to_datetime(d["t"], utc=True).dt.tz_convert(None)

    numeric_cols = ["open", "high", "low", "close", "vol_base"]
    for c in numeric_cols:
        d[c] = pd.to_numeric(d[c], errors="coerce")

    d = d.dropna(subset=["t", "open", "high", "low", "close"])
    d = d.sort_values("t")
    d = d.drop_duplicates(subset="t", keep="last")
    d = d.reset_index(drop=True)

    # Basic sanity filter. Keep volume NaN/zero handling conservative.
    valid = (
        (d["high"] >= d[["open", "close"]].max(axis=1))
        & (d["low"] <= d[["open", "close"]].min(axis=1))
        & (d["high"] >= d["low"])
        & (d["close"] > 0)
    )
    d = d.loc[valid].reset_index(drop=True)

    return d

## Resample OHLCV

In [5]:
def resample_ohlcv(
    df: pd.DataFrame,
    timeframe: str,
    time_col: str = "t",
) -> pd.DataFrame:
    """
    Resample lower timeframe OHLCV into a higher timeframe.

    Important timing convention:
    - Output t is candle OPEN time, not close time.
    - A 4h row at 00:00 represents [00:00, 04:00).
    - It becomes known only at 04:00.
    - Later merge/projector code must account for this with available_at = t + timeframe.
    """
    d = df.copy()
    d[time_col] = pd.to_datetime(d[time_col], utc=True).dt.tz_convert(None)
    d = d.sort_values(time_col).drop_duplicates(subset=time_col, keep="last")
    d = d.set_index(time_col)

    out = d.resample(
        timeframe,
        label="left",
        closed="left",
        origin="start_day",
    ).agg({
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
        "vol_base": "sum",
    })

    out = out.dropna(subset=["open", "high", "low", "close"]).reset_index()
    return out


def build_timeframes(df_1m: pd.DataFrame) -> dict[str, pd.DataFrame]:
    return {name: resample_ohlcv(df_1m, rule) for name, rule in TIMEFRAMES.items()}

## Indicator helpers

In [7]:
def wilder_rma(series: pd.Series, period: int) -> pd.Series:
    """Wilder RMA seeded by SMA(period), closer to TradingView/Pine style."""
    s = pd.to_numeric(series, errors="coerce").astype(float)
    out = pd.Series(np.nan, index=s.index, dtype=float)

    if len(s) < period:
        return out

    first_idx = period - 1
    out.iloc[first_idx] = s.iloc[:period].mean()

    for i in range(period, len(s)):
        out.iloc[i] = (out.iloc[i - 1] * (period - 1) + s.iloc[i]) / period

    return out


def smma(series: pd.Series, length: int) -> pd.Series:
    """Pine-like SMMA seeded by SMA(length)."""
    return wilder_rma(series, length)


def zlema(series: pd.Series, length: int) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce").astype(float)
    ema1 = s.ewm(span=length, adjust=False).mean()
    ema2 = ema1.ewm(span=length, adjust=False).mean()
    return ema1 + (ema1 - ema2)

def add_atr(df: pd.DataFrame, period: int = 14) -> pd.DataFrame:
    d = df.copy()

    high = d["high"].astype(float)
    low = d["low"].astype(float)
    close = d["close"].astype(float)
    prev_close = close.shift(1)

    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low - prev_close).abs(),
    ], axis=1).max(axis=1)

    d["tr"] = tr
    d["atr"] = wilder_rma(tr, period)
    d["atr_pct"] = d["atr"] / d["close"]

    return d


def add_impulse_macd(
    df: pd.DataFrame,
    ma_len: int = 34,
    signal_len: int = 9,
) -> pd.DataFrame:
    d = df.copy()

    high = d["high"].astype(float)
    low = d["low"].astype(float)
    close = d["close"].astype(float)

    hlc3 = (high + low + close) / 3.0

    imacd_hi = smma(high, ma_len)
    imacd_lo = smma(low, ma_len)
    imacd_mi = zlema(hlc3, ma_len)

    imacd_md = pd.Series(
        np.where(
            imacd_mi > imacd_hi,
            imacd_mi - imacd_hi,
            np.where(imacd_mi < imacd_lo, imacd_mi - imacd_lo, 0.0),
        ),
        index=d.index,
        dtype=float,
    )

    imacd_sb = imacd_md.rolling(signal_len, min_periods=signal_len).mean()
    imacd_sh = imacd_md - imacd_sb

    imacd_color = np.where(
        hlc3 > imacd_mi,
        np.where(hlc3 > imacd_hi, "lime", "green"),
        np.where(hlc3 < imacd_lo, "red", "orange"),
    )

    d["hlc3"] = hlc3
    d["imacd_hi"] = imacd_hi
    d["imacd_lo"] = imacd_lo
    d["imacd_mi"] = imacd_mi
    d["imacd_md"] = imacd_md
    d["imacd_sb"] = imacd_sb
    d["imacd_sh"] = imacd_sh
    d["imacd_color"] = imacd_color

    # Scale-normalized value for regime logic.
    d["imacd_sb_atr"] = d["imacd_sb"] / d["atr"].replace(0, np.nan)
    d["imacd_sh_atr"] = d["imacd_sh"] / d["atr"].replace(0, np.nan)

    return d

In [8]:
def add_features(
    df: pd.DataFrame,
    atr_period: int = 14,
    imacd_ma_len: int = 34,
    imacd_signal_len: int = 9,
) -> pd.DataFrame:
    d = add_atr(df, period=atr_period)
    d = add_impulse_macd(d, ma_len=imacd_ma_len, signal_len=imacd_signal_len)
    return d

## Regime labels

In [9]:
def label_vol_regime(
    df: pd.DataFrame,
    col: str = "atr_pct",
    lookback: int = 200,
    q_enter_high: float = 0.70,
    q_exit_high: float = 0.55,
    min_periods: int | None = None,
    out_col: str = "atr_vol_regime",
) -> pd.DataFrame:
    """
    Hysteresis volatility regime.

    No fillna here. Unknown warmup regime stays NaN.
    That prevents pretending early history is low_vol.
    """
    d = df.copy()

    if col not in d.columns:
        raise KeyError(f"Missing column: {col}")

    x = pd.to_numeric(d[col], errors="coerce")
    mp = lookback if min_periods is None else min_periods

    enter_high = x.rolling(lookback, min_periods=mp).quantile(q_enter_high)
    exit_high = x.rolling(lookback, min_periods=mp).quantile(q_exit_high)

    state = []
    current = "low_vol"

    for xi, eh, ex in zip(x, enter_high, exit_high):
        if pd.isna(xi) or pd.isna(eh) or pd.isna(ex):
            state.append(np.nan)
            continue

        if current == "low_vol" and xi > eh:
            current = "high_vol"
        elif current == "high_vol" and xi < ex:
            current = "low_vol"

        state.append(current)

    d[out_col] = state
    d[f"{out_col}_enter_high"] = enter_high
    d[f"{out_col}_exit_high"] = exit_high

    return d

In [10]:
def label_imom_regime(
    df: pd.DataFrame,
    col: str = "imacd_sb_atr",
    eps: float = 0.10,
    out_col: str = "imom_label",
) -> pd.DataFrame:
    """Impulse momentum regime using ATR-normalized IMACD signal."""
    d = df.copy()

    x = pd.to_numeric(d[col], errors="coerce")
    d[out_col] = np.where(
        x > eps,
        "plus",
        np.where(x < -eps, "minus", "flat"),
    )
    d.loc[x.isna(), out_col] = np.nan

    return d

In [11]:
def enforce_persistence(series: pd.Series, min_len: int = 3) -> pd.Series:
    """
    Confirm regime changes only after seeing the new state for min_len bars.

    This is causal: it does not rewrite past values.
    """
    s = series.copy()
    out = pd.Series(index=s.index, dtype=object)

    # Find first non-NaN state.
    first_valid_idx = s.first_valid_index()
    if first_valid_idx is None:
        return out

    first_pos = s.index.get_loc(first_valid_idx)
    current_state = s.iloc[first_pos]
    candidate_state = None
    candidate_count = 0

    out.iloc[:first_pos] = np.nan
    out.iloc[first_pos] = current_state

    for i in range(first_pos + 1, len(s)):
        x = s.iloc[i]

        if pd.isna(x):
            out.iloc[i] = np.nan
            continue

        if x == current_state:
            candidate_state = None
            candidate_count = 0
            out.iloc[i] = current_state
            continue

        if x == candidate_state:
            candidate_count += 1
        else:
            candidate_state = x
            candidate_count = 1

        if candidate_count >= min_len:
            current_state = candidate_state
            candidate_state = None
            candidate_count = 0

        out.iloc[i] = current_state

    return out

In [12]:
def add_regimes(
    df: pd.DataFrame,
    vol_lookback: int = 200,
    imom_eps_atr: float = 0.10,
    color_persistence: int | None = None,
) -> pd.DataFrame:
    d = label_vol_regime(df, lookback=vol_lookback)
    d = label_imom_regime(d, eps=imom_eps_atr)

    if color_persistence is not None:
        d["imacd_color_p"] = enforce_persistence(d["imacd_color"], min_len=color_persistence)

    return d

## Full feature/regime pipeline

In [13]:
def build_feature_regime_frames(df_1m: pd.DataFrame) -> dict[str, pd.DataFrame]:
    raw_tfs = build_timeframes(df_1m)

    out = {}
    for name, frame in raw_tfs.items():
        d = add_features(frame)

        if name in {"4h", "1d"}:
            d = add_regimes(d, color_persistence=4)
        else:
            d = add_regimes(d, color_persistence=None)

        out[name] = d

    return out

## Run

In [14]:
raw_1m = load_aggbar_years(DATA_DIR, SYMBOL, YEARS)

In [16]:
frames = build_feature_regime_frames(raw_1m)

In [17]:
df_mcf_15min = frames["15m"]
df_mcf_1h = frames["1h"]
df_mcf_4h = frames["4h"]
df_mcf_1d = frames["1d"]

# Part 2: No-leak regime merge / projection

## Merge Function

In [31]:
def tf_to_timedelta(tf: str) -> pd.Timedelta:
    """
    Convert timeframe string into pandas Timedelta.

    Supported examples:
    - "15m", "15min"
    - "1h", "4h"
    - "1d", "1D"
    """
    x = tf.strip()
    lower = x.lower()

    if lower.endswith("min"):
        return pd.Timedelta(minutes=int(lower.replace("min", "")))
    if lower.endswith("m"):
        return pd.Timedelta(minutes=int(lower.replace("m", "")))
    if lower.endswith("h"):
        return pd.Timedelta(hours=int(lower.replace("h", "")))
    if lower.endswith("d"):
        return pd.Timedelta(days=int(lower.replace("d", "")))

    raise ValueError(f"Unsupported timeframe: {tf}")

In [32]:
def add_available_at(
    df: pd.DataFrame,
    source_tf: str,
    time_col: str = "t",
    out_col: str = "available_at",
) -> pd.DataFrame:
    """
    Add the timestamp when a candle's final OHLCV-derived features become known.

    Convention:
    - t is candle-open time.
    - A 4h candle at 00:00 covers [00:00, 04:00).
    - Its high/low/close/ATR/regime are available at 04:00.
    """
    d = df.copy()
    d[time_col] = pd.to_datetime(d[time_col], utc=True).dt.tz_convert(None)
    d[out_col] = d[time_col] + tf_to_timedelta(source_tf)
    return d

In [40]:
def normalize_datetime_ns(s: pd.Series) -> pd.Series:
    """Force pandas datetime Series to timezone-naive datetime64[ns]."""
    return pd.to_datetime(s, utc=True).dt.tz_convert(None).astype("datetime64[ns]")

In [41]:
def attach_htf_regime_asof(
    base_df: pd.DataFrame,
    htf_df: pd.DataFrame,
    htf_tf: str,
    cols_to_attach: list[str],
    suffix: str,
    base_time_col: str = "t",
    htf_time_col: str = "t",
    drop_missing: bool = False,
) -> pd.DataFrame:
    """
    Attach higher-timeframe features/regimes to a lower-timeframe base frame without future leak.

    This uses htf.available_at, not htf.t.

    Example:
    - 4h row t=00:00 becomes available_at=04:00.
    - 15m rows before 04:00 cannot see it.
    - 15m row at/after 04:00 can use the latest available 4h state.

    merge_asof direction="backward" means:
    for each base timestamp, use only HTF rows with available_at <= base timestamp.
    """
    base = base_df.copy()
    htf = htf_df.copy()

    base[base_time_col] = normalize_datetime_ns(base[base_time_col])
    htf[htf_time_col] = normalize_datetime_ns(htf[htf_time_col])
    base = base.sort_values(base_time_col).reset_index(drop=True)

    htf = add_available_at(htf, source_tf=htf_tf, time_col=htf_time_col)
    htf = htf.sort_values("available_at").reset_index(drop=True)

    keep_cols = ["available_at", htf_time_col] + cols_to_attach
    htf = htf[keep_cols].copy()

    rename_map = {htf_time_col: f"source_t_{suffix}"}
    rename_map.update({c: f"{c}_{suffix}" for c in cols_to_attach})
    htf = htf.rename(columns=rename_map)

    merged = pd.merge_asof(
        base,
        htf,
        left_on=base_time_col,
        right_on="available_at",
        direction="backward",
        allow_exact_matches=True,
    )

    merged = merged.rename(columns={"available_at": f"available_at_{suffix}"})

    if drop_missing:
        attached_cols = [f"{c}_{suffix}" for c in cols_to_attach]
        merged = merged.dropna(subset=attached_cols).reset_index(drop=True)

    return merged

In [42]:
def attach_multiple_htf_regimes(
    base_df: pd.DataFrame,
    htf_specs: list[dict],
    drop_missing: bool = False,
) -> pd.DataFrame:
    """
    Attach multiple HTF regime frames sequentially.

    Each spec should contain:
    {
        "df": df_mcf_4h,
        "tf": "4h",
        "suffix": "4h",
        "cols": ["atr_vol_regime", "imom_label", "imacd_color_p"]
    }
    """
    out = base_df.copy()

    for spec in htf_specs:
        out = attach_htf_regime_asof(
            base_df=out,
            htf_df=spec["df"],
            htf_tf=spec["tf"],
            cols_to_attach=spec["cols"],
            suffix=spec["suffix"],
            drop_missing=drop_missing,
        )

    return out

In [35]:
def build_signal_frame_with_regimes(
    signal_df: pd.DataFrame,
    df_4h: pd.DataFrame,
    df_1d: pd.DataFrame,
    drop_missing_regime: bool = True,
) -> pd.DataFrame:
    """
    Build a signal timeframe frame with 4h and 1d regimes attached safely.

    Usually signal_df can be df_mcf_15min or df_mcf_1h.
    """
    htf_cols_4h = [
        "atr_vol_regime",
        "imom_label",
        "imacd_color",
        "imacd_color_p",
        "imacd_sb_atr",
        "imacd_sh_atr",
        "atr_pct",
    ]

    htf_cols_1d = [
        "atr_vol_regime",
        "imom_label",
        "imacd_color",
        "imacd_color_p",
        "imacd_sb_atr",
        "imacd_sh_atr",
        "atr_pct",
    ]

    specs = [
        {"df": df_4h, "tf": "4h", "suffix": "4h", "cols": htf_cols_4h},
        {"df": df_1d, "tf": "1d", "suffix": "1d", "cols": htf_cols_1d},
    ]

    out = attach_multiple_htf_regimes(
        base_df=signal_df,
        htf_specs=specs,
        drop_missing=drop_missing_regime,
    )

    return out

In [37]:
def assert_no_htf_future_leak(
    df: pd.DataFrame,
    base_time_col: str = "t",
    suffixes: tuple[str, ...] = ("4h", "1d"),
) -> None:
    """
    Fail loudly if any attached HTF information is used before it is available.
    """
    d = df.copy()
    d[base_time_col] = pd.to_datetime(d[base_time_col], utc=True).dt.tz_convert(None)

    for suffix in suffixes:
        available_col = f"available_at_{suffix}"
        source_col = f"source_t_{suffix}"

        if available_col not in d.columns:
            continue

        d[available_col] = pd.to_datetime(d[available_col], utc=True).dt.tz_convert(None)

        bad_available = d[available_col].notna() & (d[available_col] > d[base_time_col])
        if bad_available.any():
            examples = d.loc[bad_available, [base_time_col, source_col, available_col]].head()
            msg = (
                f"Future leak detected for {suffix}: available_at > base t"
                f"{examples}"
            )
            raise AssertionError(msg)

    print("No HTF future leak detected.")

In [38]:
def print_htf_merge_sample(
    df: pd.DataFrame,
    suffix: str = "4h",
    n: int = 20,
) -> None:
    """Small debugging view for checking HTF attachment timing."""
    cols = [
        "t",
        f"source_t_{suffix}",
        f"available_at_{suffix}",
        f"atr_vol_regime_{suffix}",
        f"imom_label_{suffix}",
        f"imacd_color_p_{suffix}",
    ]
    existing_cols = [c for c in cols if c in df.columns]
    print(df[existing_cols].tail(n))

## Run

In [43]:
df_signal_15m = build_signal_frame_with_regimes(
    signal_df=df_mcf_15min,
    df_4h=df_mcf_4h,
    df_1d=df_mcf_1d,
    drop_missing_regime=True,
)

In [44]:
assert_no_htf_future_leak(df_signal_15m)

No HTF future leak detected.


In [48]:
df_signal_1h = build_signal_frame_with_regimes(
    signal_df=df_mcf_1h,
    df_4h=df_mcf_4h,
    df_1d=df_mcf_1d,
    drop_missing_regime=True,
)

In [49]:
assert_no_htf_future_leak(df_signal_1h)

No HTF future leak detected.


# Part 3: Backtest

## Function & Strategies

In [63]:
import numpy as np
import pandas as pd
import itertools


STRATEGY_IMPORTANT_COLS = {
    "donchian_breakout": [
        "open", "high", "low", "close",
        "donchian_entry_hi", "donchian_entry_lo",
        "donchian_exit_hi", "donchian_exit_lo",
    ],

    "atr_breakout": [
        "open", "high", "low", "close",
        "atr",
    ],

    "kc_breakout": [
        "open", "high", "low", "close",
        "kc_upper", "kc_lower", "kc_mid",
        "atr",
    ],

    "ma_cross": [
        "open", "high", "low", "close",
        "ema_10", "ema_20", "ema_50", "ema_100",
        "sma_10", "sma_20", "sma_50", "sma_100",
    ],

    "triple_ma": [
        "open", "high", "low", "close",
        "ema_10", "ema_20", "ema_50", "ema_100",
        "sma_10", "sma_20", "sma_50", "sma_100",
    ],

    "price_vs_ma": [
        "open", "high", "low", "close",
        "ema_20", "ema_50", "ema_100",
        "sma_20", "sma_50", "sma_100",
    ],

    "macd_trend": [
        "open", "high", "low", "close",
        "macd", "macd_signal", "macd_hist",
        "atr",
    ],

    "ts_momentum": [
        "open", "high", "low", "close",
    ],

    "roc_momentum": [
        "open", "high", "low", "close",
    ],

    "linreg_slope": [
        "open", "high", "low", "close",
    ],

    "bb_expansion_trend": [
        "open", "high", "low", "close",
        "bb_upper", "bb_lower", "bb_mid", "bb_width", "bb_percent_b",
    ],

    "rsi_mr": [
        "open", "high", "low", "close",
        "rsi",
    ],

    "bb_mr": [
        "open", "high", "low", "close",
        "bb_upper", "bb_lower", "bb_mid", "bb_width", "bb_percent_b",
    ],

    "percentb_mr": [
        "open", "high", "low", "close",
        "bb_percent_b", "bb_upper", "bb_lower", "bb_mid", "bb_width",
    ],

    "kc_mr": [
        "open", "high", "low", "close",
        "kc_upper", "kc_lower", "kc_mid",
        "atr",
    ],

    "stochrsi_mr": [
        "open", "high", "low", "close",
        "rsi",
        "stoch_rsi_k", "stoch_rsi_d",
        "stoch_rsi_k_100", "stoch_rsi_d_100",
    ],

    "vwap_fade": [
        "open", "high", "low", "close", "volume",
        "vwap", "atr",
    ],

    "zscore_mr": [
        "open", "high", "low", "close",
    ],
}

In [64]:
import itertools
import math
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd

# =========================================================
# 1) Config / helpers
# =========================================================

@dataclass
class BTConfig:
    fee_bps: float = 1.0
    slippage_bps: float = 1.0
    max_leverage: float = 3.0
    vol_target: float | None = None
    bars_per_year: int = 365 * 24
    embargo_bars: int = 0
    allow_short: bool = True


def tf_to_timedelta(tf: str) -> pd.Timedelta:
    tf = str(tf).lower().strip()
    mapping = {
        "1m": pd.Timedelta(minutes=1),
        "5m": pd.Timedelta(minutes=5),
        "15m": pd.Timedelta(minutes=15),
        "30m": pd.Timedelta(minutes=30),
        "1h": pd.Timedelta(hours=1),
        "4h": pd.Timedelta(hours=4),
        "1d": pd.Timedelta(days=1),
        "1day": pd.Timedelta(days=1),
        "1min": pd.Timedelta(minutes=1),
    }
    if tf not in mapping:
        raise ValueError(f"Unsupported timeframe: {tf}")
    return mapping[tf]


def _cost_from_turnover_np(turnover: np.ndarray, cfg: BTConfig) -> np.ndarray:
    bps = (cfg.fee_bps + cfg.slippage_bps) * 1e-4
    return turnover * bps


def compute_metrics_np(equity: np.ndarray, rets: np.ndarray, cfg: BTConfig) -> dict:
    rets = np.asarray(rets, dtype=np.float64)
    equity = np.asarray(equity, dtype=np.float64)

    valid_r = np.isfinite(rets)
    valid_e = np.isfinite(equity)

    rets = rets[valid_r]
    equity = equity[valid_e]

    if rets.size == 0 or equity.size == 0:
        return {
            "sharpe": np.nan,
            "maxdd": np.nan,
            "cagr": np.nan,
            "ann_vol": np.nan,
            "ann_return": np.nan,
            "terminal_equity": np.nan,
            "turnover_mean": np.nan,
            "hit_rate": np.nan,
            "avg_gross": np.nan,
            "avg_cost": np.nan,
            "pnl_sum": np.nan,
        }

    ann = cfg.bars_per_year
    mu = rets.mean() * ann
    vol = rets.std(ddof=0) * np.sqrt(ann)
    sharpe = np.nan if vol == 0 else mu / vol

    peak = np.maximum.accumulate(equity)
    dd = equity / peak - 1.0
    maxdd = np.min(dd)

    years = len(equity) / ann
    cagr = (equity[-1] / equity[0]) ** (1 / years) - 1 if years > 0 and equity[0] > 0 else np.nan

    return {
        "sharpe": sharpe,
        "maxdd": maxdd,
        "cagr": cagr,
        "ann_vol": vol,
        "ann_return": mu,
        "terminal_equity": equity[-1],
    }


def safe_metric(metrics, key, default=np.nan):
    return metrics[key] if key in metrics else default


def freeze_params(params: dict) -> Tuple[Tuple[str, Any], ...]:
    """
    แปลง dict params เป็น hashable key สำหรับ cache
    """
    return tuple(sorted(params.items(), key=lambda x: x[0]))


# =========================================================
# 2) Stop-loss on HTF signal
# =========================================================

def apply_stop_loss_close_fast(sig_raw, close, stop_pct=0.02) -> np.ndarray:
    """
    Logic เดิม:
      - state = {-1,0,1}
      - long: stop ถ้า px <= entry*(1-stop_pct)
      - short: stop ถ้า px >= entry*(1+stop_pct)
      - ถ้า desired เปลี่ยนจาก state เดิม -> flat
    """
    sig_raw = np.asarray(sig_raw, dtype=np.float64)
    close = np.asarray(close, dtype=np.float64)

    n = len(sig_raw)
    out = np.zeros(n, dtype=np.float64)

    state = 0.0
    entry_price = np.nan

    for i in range(n):
        desired = 0.0 if not np.isfinite(sig_raw[i]) else sig_raw[i]
        px = close[i]

        if state == 0.0:
            if desired == 1.0:
                state = 1.0
                entry_price = px
            elif desired == -1.0:
                state = -1.0
                entry_price = px

        elif state == 1.0:
            stop_hit = px <= entry_price * (1.0 - stop_pct)
            if stop_hit:
                state = 0.0
                entry_price = np.nan
            elif desired != 1.0:
                state = 0.0
                entry_price = np.nan

        elif state == -1.0:
            stop_hit = px >= entry_price * (1.0 + stop_pct)
            if stop_hit:
                state = 0.0
                entry_price = np.nan
            elif desired != -1.0:
                state = 0.0
                entry_price = np.nan

        out[i] = state

    return out


# =========================================================
# 3) Condition utilities
# =========================================================

def make_condition_dicts(grouping_dict, allowed_values):
    out = {}
    for group_name, cols in grouping_dict.items():
        combos = itertools.product(*(allowed_values[c] for c in cols))
        out[group_name] = [dict(zip(cols, vals)) for vals in combos]
    return out


def precompute_condition_masks(df_signal: pd.DataFrame, grouping_dict, allowed_values):
    """
    เร็วกว่า build_mask_from_conditions ทีละรอบเยอะ
    โดย cast แต่ละ regime col เป็น string แค่ครั้งเดียว
    """
    all_group_conditions = make_condition_dicts(grouping_dict, allowed_values)

    str_cols = {}
    for col in allowed_values.keys():
        if col in df_signal.columns:
            # ใช้ object/string แบบเดียวกันทั้งคอลัมน์
            arr = df_signal[col].astype("string").fillna("<NA>").to_numpy()
            str_cols[col] = arr

    mask_map = {}
    meta_rows = []

    n = len(df_signal)

    for group_name, conds in all_group_conditions.items():
        for cond in conds:
            mask = np.ones(n, dtype=bool)
            for col, val in cond.items():
                mask &= (str_cols[col] == str(val))

            cond_str = " & ".join([f"{k}={v}" for k, v in cond.items()])
            key = (group_name, cond_str)

            mask_map[key] = mask
            meta_rows.append({
                "group_name": group_name,
                "group_cols": tuple(grouping_dict[group_name]),
                "condition_dict": cond,
                "condition_str": cond_str,
                "n_conditions": len(cond),
                "n_active_signal_bars": int(mask.sum()),
                "active_ratio_signal": float(mask.mean()) if n > 0 else 0.0,
            })

    meta_df = pd.DataFrame(meta_rows)
    return all_group_conditions, mask_map, meta_df


# =========================================================
# 4) Fast projector: HTF decision -> exec timeline
# =========================================================

class HTFExecProjector:
    """
    precompute mapping:
      exec bar t_exec -> latest available signal index j
    where available_at[j] = signal_t[j] + signal_tf
    """
    def __init__(self, df_exec: pd.DataFrame, df_signal: pd.DataFrame, signal_tf: str):
        self.df_exec = df_exec.copy().reset_index(drop=True)
        self.df_signal = df_signal.copy().reset_index(drop=True)

        self.df_exec["t"] = pd.to_datetime(self.df_exec["t"]).astype("datetime64[ns]")
        self.df_signal["t"] = pd.to_datetime(self.df_signal["t"]).astype("datetime64[ns]")

        self.exec_t = self.df_exec["t"].to_numpy(dtype="datetime64[ns]")
        self.signal_t = self.df_signal["t"].to_numpy(dtype="datetime64[ns]")
        self.available_at = (self.df_signal["t"] + tf_to_timedelta(signal_tf)).to_numpy(dtype="datetime64[ns]")

        # map exec_t -> latest available signal idx
        # searchsorted(..., side='right') - 1
        idx = np.searchsorted(self.available_at, self.exec_t, side="right") - 1
        self.exec_to_signal_idx = idx.astype(np.int64)

        self.close_exec = self.df_exec["close"].to_numpy(dtype=np.float64)
        self.ret_exec = pd.Series(self.close_exec).pct_change().fillna(0.0).to_numpy(dtype=np.float64)

    def project_known_arrays(
        self,
        signal_htf: np.ndarray,
        active_htf: np.ndarray,
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        project signal/active จาก HTF ลง exec bars
        """
        n_exec = len(self.exec_to_signal_idx)

        known_signal = np.zeros(n_exec, dtype=np.float64)
        known_active = np.zeros(n_exec, dtype=np.int8)

        valid = self.exec_to_signal_idx >= 0
        idx = self.exec_to_signal_idx[valid]

        known_signal[valid] = signal_htf[idx]
        known_active[valid] = active_htf[idx]

        return known_signal, known_active


# =========================================================
# 5) Fast backtest engine
# =========================================================

def _shift_with_fill(arr: np.ndarray, periods: int, fill_value=0.0) -> np.ndarray:
    arr = np.asarray(arr)
    out = np.empty_like(arr, dtype=np.float64 if arr.dtype.kind in "fc" else arr.dtype)
    if periods <= 0:
        out[:] = arr
        return out
    out[:periods] = fill_value
    out[periods:] = arr[:-periods]
    return out


def _rolling_std_shifted(arr: np.ndarray, window: int) -> np.ndarray:
    """
    ใช้ pandas ทีเดียวพอ เพราะอยู่นอก condition loop
    """
    return (
        pd.Series(arr)
        .rolling(window)
        .std(ddof=0)
        .shift(1)
        .to_numpy(dtype=np.float64)
    )


def build_param_signal_cache(
    df_signal: pd.DataFrame,
    strat,
    param_grid: List[dict],
    cfg: BTConfig,
    stop_pct: float | None = None,
) -> Dict[Tuple[Tuple[str, Any], ...], np.ndarray]:
    """
    cache signal ต่อ param set
    """
    cache = {}
    close_signal = df_signal["close"].to_numpy(dtype=np.float64)

    for params in param_grid:
        pkey = freeze_params(params)
        if pkey in cache:
            continue

        sig_raw = strat.signal(df_signal, **params)
        if isinstance(sig_raw, pd.Series):
            sig_raw = sig_raw.to_numpy(dtype=np.float64)
        else:
            sig_raw = np.asarray(sig_raw, dtype=np.float64)

        sig_raw = np.nan_to_num(sig_raw, nan=0.0)

        if stop_pct is not None:
            sig_raw = apply_stop_loss_close_fast(sig_raw, close_signal, stop_pct=stop_pct)

        if not cfg.allow_short:
            sig_raw = np.clip(sig_raw, 0.0, None)

        cache[pkey] = sig_raw.astype(np.float64, copy=False)

    return cache


def run_backtest_fast_from_precomputed(
    projector: HTFExecProjector,
    sig_raw_htf: np.ndarray,
    cfg: BTConfig,
    active_mask_signal: np.ndarray | None = None,
    force_flat_outside: bool = True,
    exec_delay_bars: int = 1,
    vol_lookback: int | None = None,
    return_bt_df: bool = True,
):
    n_sig = len(sig_raw_htf)
    n_exec = len(projector.df_exec)

    if active_mask_signal is None:
        active_mask_signal = np.ones(n_sig, dtype=bool)
    else:
        active_mask_signal = np.asarray(active_mask_signal, dtype=bool)

    sig_effective = np.where(active_mask_signal, sig_raw_htf, 0.0)
    active_htf = active_mask_signal.astype(np.int8)

    known_signal, known_active = projector.project_known_arrays(sig_effective, active_htf)

    pos = _shift_with_fill(known_signal, exec_delay_bars, fill_value=0.0).astype(np.float64)
    active_exec = _shift_with_fill(known_active.astype(np.float64), exec_delay_bars, fill_value=0.0).astype(np.int8)

    lev = np.ones(n_exec, dtype=np.float64)

    if cfg.vol_target is not None:
        lookback = 50 if vol_lookback is None else int(vol_lookback)
        rv = _rolling_std_shifted(projector.ret_exec, lookback) * np.sqrt(cfg.bars_per_year)
        with np.errstate(divide="ignore", invalid="ignore"):
            lev = cfg.vol_target / rv
        lev = np.clip(lev, 0.0, cfg.max_leverage)
        lev = np.nan_to_num(lev, nan=0.0, posinf=cfg.max_leverage, neginf=0.0)

    pos = np.clip(pos * lev, -cfg.max_leverage, cfg.max_leverage)

    if force_flat_outside:
        pos = np.where(active_exec == 1, pos, 0.0)

    ret_ = projector.ret_exec
    gross = pos * ret_

    pos_prev = _shift_with_fill(pos, 1, fill_value=0.0)
    turnover = np.abs(pos - pos_prev)
    cost = _cost_from_turnover_np(turnover, cfg)

    net = gross - cost
    equity = np.cumprod(1.0 + net)

    metrics = compute_metrics_np(equity, net, cfg)
    metrics["turnover_mean"] = turnover.mean()
    metrics["hit_rate"] = (net > 0).mean()
    metrics["avg_gross"] = gross.mean()
    metrics["avg_cost"] = cost.mean()
    metrics["pnl_sum"] = net.sum()

    if not return_bt_df:
        return None, metrics

    out = projector.df_exec.copy()
    out["signal_known"] = known_signal
    out["active_known"] = known_active
    out["pos"] = pos
    out["ret"] = ret_
    out["gross"] = gross
    out["cost"] = cost
    out["net"] = net
    out["equity"] = equity
    out["turnover"] = turnover
    out["lev"] = lev
    out["active_exec"] = active_exec
    return out, metrics


# =========================================================
# 6) Compress exec bt -> signal bt
#    (ยังใช้ pandas ได้ เพราะไม่ได้อยู่ใน inner loop สำคัญ)
# =========================================================

def compress_exec_to_signal_bt_fast(
    bt_exec_df: pd.DataFrame,
    df_signal: pd.DataFrame,
    keep_signal_cols=None,
):
    sig = df_signal.copy().reset_index(drop=True)
    ex = bt_exec_df.copy().reset_index(drop=True)

    sig["t"] = pd.to_datetime(sig["t"]).astype("datetime64[ns]")
    ex["t"] = pd.to_datetime(ex["t"]).astype("datetime64[ns]")

    signal_t = sig["t"].to_numpy(dtype="datetime64[ns]")
    exec_t = ex["t"].to_numpy(dtype="datetime64[ns]")

    idx = np.searchsorted(signal_t, exec_t, side="right") - 1
    ex["signal_idx"] = idx

    valid = ex["signal_idx"] >= 0
    ex2 = ex.loc[valid].copy()
    ex2["signal_bar_t"] = sig["t"].to_numpy()[ex2["signal_idx"].to_numpy()]

    agg_dict = {}
    if "gross" in ex2.columns:
        agg_dict["gross_pnl"] = ("gross", "sum")
    if "cost" in ex2.columns:
        agg_dict["cost_pnl"] = ("cost", "sum")
    if "net" in ex2.columns:
        agg_dict["net_pnl"] = ("net", "sum")
    if "turnover" in ex2.columns:
        agg_dict["turnover_sum"] = ("turnover", "sum")
        agg_dict["turnover_mean_exec"] = ("turnover", "mean")
    if "equity" in ex2.columns:
        agg_dict["equity"] = ("equity", "last")
    if "active_exec" in ex2.columns:
        agg_dict["active_exec_any"] = ("active_exec", "max")
        agg_dict["active_exec_share"] = ("active_exec", "mean")
    if "signal_known" in ex2.columns:
        agg_dict["signal_known_first"] = ("signal_known", "first")
        agg_dict["signal_known_last"] = ("signal_known", "last")

    agg_dict["n_exec_bars"] = ("t", "size")

    ex_agg = (
        ex2.groupby("signal_bar_t", dropna=False)
        .agg(**agg_dict)
        .reset_index()
        .rename(columns={"signal_bar_t": "t"})
    )

    if keep_signal_cols is None:
        base_cols = list(sig.columns)
    else:
        base_cols = ["t"] + [c for c in keep_signal_cols if c in sig.columns and c != "t"]

    out = sig[base_cols].merge(ex_agg, on="t", how="left")

    for c in ["gross_pnl", "cost_pnl", "net_pnl", "turnover_sum", "turnover_mean_exec"]:
        if c in out.columns:
            out[c] = out[c].fillna(0.0)

    for c in ["active_exec_any", "active_exec_share", "n_exec_bars"]:
        if c in out.columns:
            out[c] = out[c].fillna(0)

    if "net_pnl" in out.columns:
        out["signal_bar_ret"] = out["net_pnl"]

    return out


# =========================================================
# 7) Keep columns helper
# =========================================================

def get_keep_signal_cols_for_strategy(
    strat,
    df_signal,
    allowed_values,
    extra_keep_cols=None,
    include_ohlcv=True,
):
    strategy_name = getattr(strat, "name", strat.__class__.__name__)

    cols = ["t"]

    if include_ohlcv:
        for c in ["open", "high", "low", "close", "volume"]:
            if c in df_signal.columns:
                cols.append(c)

    regime_cols = [c for c in allowed_values.keys() if c in df_signal.columns]
    cols.extend(regime_cols)

    strat_cols = STRATEGY_IMPORTANT_COLS.get(strategy_name, [])
    cols.extend([c for c in strat_cols if c in df_signal.columns])

    if extra_keep_cols is not None:
        cols.extend([c for c in extra_keep_cols if c in df_signal.columns])

    seen = set()
    out = []
    for c in cols:
        if c not in seen:
            seen.add(c)
            out.append(c)

    return out


# =========================================================
# 8) Main scanner (metrics only / store bt)
# =========================================================

def scan_groupings_fast(
    df_exec,
    df_signal,
    signal_tf,
    strat,
    param_grid,
    cfg,
    grouping_dict,
    allowed_values,
    stop_pct=None,
    force_flat_outside=True,
    exec_delay_bars=1,
    strategy_name=None,
    extra_meta=None,
    keep_signal_cols=None,
    store_bt=False,
    store_key_mode="run_id",
    verbose=True,
):
    """
    เวอร์ชันเร็ว:
      - precompute condition masks ครั้งเดียว
      - precompute HTF->exec mapping ครั้งเดียว
      - cache signal per param
      - run inner loop ด้วย numpy
    """
    if strategy_name is None:
        strategy_name = getattr(strat, "name", strat.__class__.__name__)

    projector = HTFExecProjector(df_exec=df_exec, df_signal=df_signal, signal_tf=signal_tf)

    _, mask_map, cond_meta_df = precompute_condition_masks(
        df_signal=df_signal,
        grouping_dict=grouping_dict,
        allowed_values=allowed_values,
    )

    signal_cache = build_param_signal_cache(
        df_signal=df_signal,
        strat=strat,
        param_grid=param_grid,
        cfg=cfg,
        stop_pct=stop_pct,
    )

    rows = []
    bt_signal_dict = {}
    run_id = 0
    total_runs = len(param_grid) * len(cond_meta_df)
    done = 0

    for p_idx, params in enumerate(param_grid):
        pkey = freeze_params(params)
        sig_raw = signal_cache[pkey]
        vol_lookback = params.get("vol_lookback", 50)

        for _, meta in cond_meta_df.iterrows():
            group_name = meta["group_name"]
            cond_str = meta["condition_str"]
            cond = meta["condition_dict"]
            group_cols = meta["group_cols"]

            active_mask_signal = mask_map[(group_name, cond_str)]
            n_active = int(meta["n_active_signal_bars"])
            active_ratio = float(meta["active_ratio_signal"])

            if n_active == 0:
                bt_exec_df = None
                metrics = {}
            else:
                bt_exec_df, metrics = run_backtest_fast_from_precomputed(
                    projector=projector,
                    sig_raw_htf=sig_raw,
                    cfg=cfg,
                    active_mask_signal=active_mask_signal,
                    force_flat_outside=force_flat_outside,
                    exec_delay_bars=exec_delay_bars,
                    vol_lookback=vol_lookback,
                    return_bt_df=store_bt,
                )

            row = {
                "run_id": run_id,
                "strategy_name": strategy_name,
                "signal_tf": signal_tf,
                "group_name": group_name,
                "group_cols": tuple(group_cols),
                "condition_dict": cond,
                "condition_str": cond_str,
                "n_conditions": len(cond),
                "param_id": p_idx,
                "params": params,
                "n_active_signal_bars": n_active,
                "active_ratio_signal": active_ratio,
                "sharpe": safe_metric(metrics, "sharpe"),
                "cagr": safe_metric(metrics, "cagr"),
                "maxdd": safe_metric(metrics, "maxdd"),
                "ann_vol": safe_metric(metrics, "ann_vol"),
                "ann_return": safe_metric(metrics, "ann_return"),
                "terminal_equity": safe_metric(metrics, "terminal_equity"),
                "turnover_mean": safe_metric(metrics, "turnover_mean"),
                "hit_rate": safe_metric(metrics, "hit_rate"),
                "avg_gross": safe_metric(metrics, "avg_gross"),
                "avg_cost": safe_metric(metrics, "avg_cost"),
                "pnl_sum": safe_metric(metrics, "pnl_sum"),
            }

            for c in allowed_values.keys():
                row[c] = cond.get(c, np.nan)

            if extra_meta is not None:
                row.update(extra_meta)

            rows.append(row)

            if store_bt and bt_exec_df is not None:
                bt_signal_df = compress_exec_to_signal_bt_fast(
                    bt_exec_df=bt_exec_df,
                    df_signal=df_signal,
                    keep_signal_cols=keep_signal_cols,
                )

                bt_signal_df = bt_signal_df.copy()
                bt_signal_df["run_id"] = run_id
                bt_signal_df["strategy_name"] = strategy_name
                bt_signal_df["group_name"] = group_name
                bt_signal_df["condition_str"] = cond_str
                bt_signal_df["params"] = str(params)

                if store_key_mode == "run_id":
                    key = run_id
                elif store_key_mode == "tuple":
                    key = (strategy_name, p_idx, group_name, cond_str)
                else:
                    raise ValueError("store_key_mode must be 'run_id' or 'tuple'")

                bt_signal_dict[key] = bt_signal_df

            run_id += 1
            done += 1

            if verbose and (done % 25 == 0 or done == total_runs):
                print(f"scan progress: {done}/{total_runs}")

    summary_df = pd.DataFrame(rows)
    return summary_df, bt_signal_dict


# =========================================================
# 9) Multiple strategies
# =========================================================

def scan_multiple_strategies_fast(
    df_exec,
    df_signal,
    signal_tf,
    strategy_param_list,
    cfg,
    grouping_dict,
    allowed_values,
    stop_pct=None,
    force_flat_outside=True,
    exec_delay_bars=1,
    extra_meta=None,
    extra_keep_cols=None,
    include_ohlcv=True,
    store_bt=False,
    store_key_mode="run_id",
    verbose=True,
):
    summary_parts = []
    bt_signal_dict = {}
    global_run_id = 0

    for strat, param_grid in strategy_param_list:
        strategy_name = getattr(strat, "name", strat.__class__.__name__)

        keep_signal_cols = get_keep_signal_cols_for_strategy(
            strat=strat,
            df_signal=df_signal,
            allowed_values=allowed_values,
            extra_keep_cols=extra_keep_cols,
            include_ohlcv=include_ohlcv,
        )

        if verbose:
            print(f"\n=== scanning strategy: {strategy_name} ===")
            print("keep_signal_cols:", keep_signal_cols)

        part_summary_df, part_bt_signal_dict = scan_groupings_fast(
            df_exec=df_exec,
            df_signal=df_signal,
            signal_tf=signal_tf,
            strat=strat,
            param_grid=param_grid,
            cfg=cfg,
            grouping_dict=grouping_dict,
            allowed_values=allowed_values,
            stop_pct=stop_pct,
            force_flat_outside=force_flat_outside,
            exec_delay_bars=exec_delay_bars,
            strategy_name=strategy_name,
            extra_meta=extra_meta,
            keep_signal_cols=keep_signal_cols,
            store_bt=store_bt,
            store_key_mode="run_id",
            verbose=verbose,
        )

        old_to_new = {}
        for old_run_id in part_summary_df["run_id"].tolist():
            old_to_new[int(old_run_id)] = global_run_id
            global_run_id += 1

        part_summary_df = part_summary_df.copy()
        part_summary_df["run_id"] = part_summary_df["run_id"].map(old_to_new)

        if store_bt:
            for k, v in part_bt_signal_dict.items():
                new_k = old_to_new[int(k)]
                v2 = v.copy()
                v2["run_id"] = new_k
                bt_signal_dict[new_k] = v2

        summary_parts.append(part_summary_df)

    if len(summary_parts) == 0:
        return pd.DataFrame(), {}

    summary_df = pd.concat(summary_parts, axis=0, ignore_index=True)
    return summary_df, bt_signal_dict

In [65]:
def attach_known_state_asof(
    df_target: pd.DataFrame,
    df_source: pd.DataFrame,
    source_tf: str,
    cols: list[str],
    target_time_col: str = "t",
    source_time_col: str = "t",
    prefix: str = "",
) -> pd.DataFrame:
    """
    เอา state/signal จาก source timeframe ไปแปะบน target timeframe
    โดยถือว่า source bar ที่เวลา t จะ 'รู้ได้' ตอน t + source_tf
    """

    out = df_target.copy().sort_values(target_time_col).reset_index(drop=True)
    src = df_source.copy().sort_values(source_time_col).reset_index(drop=True)

    out[target_time_col] = pd.to_datetime(out[target_time_col]).astype("datetime64[ns]")
    src[source_time_col] = pd.to_datetime(src[source_time_col]).astype("datetime64[ns]")

    src["available_at"] = src[source_time_col] + tf_to_timedelta(source_tf)

    keep = ["available_at"] + cols
    if prefix:
        rename_map = {c: f"{prefix}{c}" for c in cols}
        src = src[keep].rename(columns=rename_map)
        keep = ["available_at"] + [rename_map[c] for c in cols]
    else:
        src = src[keep]

    merged = pd.merge_asof(
        out.sort_values(target_time_col),
        src.sort_values("available_at"),
        left_on=target_time_col,
        right_on="available_at",
        direction="backward",
        allow_exact_matches=True,
    )

    return merged.drop(columns=["available_at"])

In [66]:
def attach_regime_to_signal_df(
    df_signal: pd.DataFrame,
    signal_tf: str,
    df_regime: pd.DataFrame | None = None,
    regime_tf: str | None = None,
    regime_cols: list[str] | None = None,
) -> pd.DataFrame:
    """
    ถ้า regime_tf == signal_tf และ df_regime คือ dataframe เดียวกัน
    ก็ใช้ regime columns ตรง ๆ ได้
    ถ้าต่าง timeframe จะ attach แบบ available_at
    """
    if df_regime is None or regime_cols is None or regime_tf is None:
        return df_signal.copy()

    if regime_tf == signal_tf:
        return df_signal.copy()

    signal_base_cols = [c for c in df_signal.columns if c not in regime_cols]
    signal_base = df_signal[signal_base_cols].copy()

    out = attach_known_state_asof(
        df_target=signal_base,
        df_source=df_regime,
        source_tf=regime_tf,
        cols=regime_cols,
        target_time_col="t",
        source_time_col="t",
        prefix="",
    )
    return out

In [67]:
import numpy as np
import pandas as pd

class Strategy:
    name = "base"
    def signal(self, df: pd.DataFrame, **params) -> pd.Series:
        raise NotImplementedError


def stateful_signal_from_entries_exits(
    long_entry: pd.Series,
    long_exit: pd.Series,
    short_entry: pd.Series,
    short_exit: pd.Series,
    allow_flip: bool = True,
) -> pd.Series:
    """
    Returns target position series in {-1,0,1}.
    Signal is decided at bar t, execution should still be handled by caller via shift(1).
    """
    idx = long_entry.index
    out = pd.Series(0.0, index=idx)
    state = 0

    for i in range(len(idx)):
        le = bool(long_entry.iat[i]) if pd.notna(long_entry.iat[i]) else False
        lx = bool(long_exit.iat[i]) if pd.notna(long_exit.iat[i]) else False
        se = bool(short_entry.iat[i]) if pd.notna(short_entry.iat[i]) else False
        sx = bool(short_exit.iat[i]) if pd.notna(short_exit.iat[i]) else False

        if state == 0:
            if le and not se:
                state = 1
            elif se and not le:
                state = -1

        elif state == 1:
            if lx:
                state = 0
            if allow_flip and se:
                state = -1

        elif state == -1:
            if sx:
                state = 0
            if allow_flip and le:
                state = 1

        out.iat[i] = state

    return out


def sign_with_deadband(x: pd.Series, entry: float = 0.0, exit: float | None = None) -> pd.Series:
    """
    Stateful sign with hysteresis:
    - enter long if x >= entry
    - enter short if x <= -entry
    - if holding long, exit when x <= exit
    - if holding short, exit when x >= -exit
    """
    if exit is None:
        exit = entry

    out = pd.Series(0.0, index=x.index)
    state = 0

    for i in range(len(x)):
        xi = x.iat[i]
        if pd.isna(xi):
            out.iat[i] = state
            continue

        if state == 0:
            if xi >= entry:
                state = 1
            elif xi <= -entry:
                state = -1
        elif state == 1:
            if xi <= exit:
                state = 0
        elif state == -1:
            if xi >= -exit:
                state = 0

        out.iat[i] = state

    return out.astype(float)


class DonchianBreakout(Strategy):
    name = "donchian_breakout"

    def signal(self, df, lookback=20, exit_lookback=10, allow_flip=True):
        entry_hi = df["high"].rolling(lookback).max().shift(1)
        entry_lo = df["low"].rolling(lookback).min().shift(1)

        exit_hi = df["high"].rolling(exit_lookback).max().shift(1)
        exit_lo = df["low"].rolling(exit_lookback).min().shift(1)

        long_entry = df["close"] > entry_hi
        short_entry = df["close"] < entry_lo

        long_exit = df["close"] < exit_lo
        short_exit = df["close"] > exit_hi

        return stateful_signal_from_entries_exits(
            long_entry, long_exit, short_entry, short_exit, allow_flip=allow_flip
        )

class ATRBreakout(Strategy):
    name = "atr_breakout"

    def signal(self, df, k=1.0, exit_k=0.0, allow_flip=True):
        ref = df["close"].shift(1)
        atr = df["atr"].shift(1)

        up = ref + k * atr
        dn = ref - k * atr

        long_entry = df["close"] > up
        short_entry = df["close"] < dn

        # exit when price comes back through prior close +/- exit_k*atr
        long_exit = df["close"] < (ref + exit_k * atr)
        short_exit = df["close"] > (ref - exit_k * atr)

        return stateful_signal_from_entries_exits(
            long_entry, long_exit, short_entry, short_exit, allow_flip=allow_flip
        )

class KeltnerBreakout(Strategy):
    name = "kc_breakout"

    def signal(self, df, allow_flip=True):
        upper = df["kc_upper"].shift(1)
        lower = df["kc_lower"].shift(1)
        mid = df["kc_mid"].shift(1)

        long_entry = df["close"] > upper
        short_entry = df["close"] < lower

        long_exit = df["close"] < mid
        short_exit = df["close"] > mid

        return stateful_signal_from_entries_exits(
            long_entry, long_exit, short_entry, short_exit, allow_flip=allow_flip
        )

class MACross(Strategy):
    name = "ma_cross"

    def signal(self, df, fast=10, slow=50, ma_type="ema", entry=0.0, exit=None, normalize=True):
        f = df[f"{ma_type}_{fast}"]
        s = df[f"{ma_type}_{slow}"]

        x = f - s
        if normalize:
            x = x / df["close"].replace(0, np.nan)

        return sign_with_deadband(x.fillna(0.0), entry=entry, exit=exit)

class TripleMA(Strategy):
    name = "triple_ma"

    def signal(self, df, fast=10, mid=20, slow=50, ma_type="ema"):
        f = df[f"{ma_type}_{fast}"]
        m = df[f"{ma_type}_{mid}"]
        s = df[f"{ma_type}_{slow}"]

        long_entry = (f > m) & (m > s)
        short_entry = (f < m) & (m < s)

        long_exit = ~(f > m)
        short_exit = ~(f < m)

        return stateful_signal_from_entries_exits(
            long_entry, long_exit, short_entry, short_exit, allow_flip=True
        )

class PriceVsMA(Strategy):
    name = "price_vs_ma"

    def signal(self, df, ma=100, ma_type="ema", entry=0.002, exit=0.0):
        m = df[f"{ma_type}_{ma}"]
        x = (df["close"] / m) - 1.0
        return sign_with_deadband(x.fillna(0.0), entry=entry, exit=exit)

class MACDTrend(Strategy):
    name = "macd_trend"

    def signal(self, df, entry=0.0, exit=None, normalize=False):
        x = df["macd"] - df["macd_signal"]
        if normalize and "atr" in df:
            x = x / df["atr"].replace(0, np.nan)
        return sign_with_deadband(x.fillna(0.0), entry=entry, exit=exit)

class TSMomentum(Strategy):
    name = "ts_momentum"

    def signal(self, df, lookback=20, entry=0.01, exit=0.0):
        mom = df["close"].pct_change(lookback)
        return sign_with_deadband(mom.fillna(0.0), entry=entry, exit=exit)


class ROCMomentum(Strategy):
    name = "roc_momentum"

    def signal(self, df, lookback=20, entry=0.01, exit=0.0):
        roc = df["close"] / df["close"].shift(lookback) - 1.0
        return sign_with_deadband(roc.fillna(0.0), entry=entry, exit=exit)

class LinRegSlopeTrend(Strategy):
    name = "linreg_slope"

    def signal(self, df, lookback=50, entry=0.0, exit=None, normalize=True):
        y = df["close"]
        x = np.arange(lookback)

        def slope(arr):
            xx = x
            yy = arr
            xxm = xx.mean()
            yym = yy.mean()
            denom = ((xx - xxm) ** 2).sum()
            if denom == 0:
                return 0.0
            return ((xx - xxm) * (yy - yym)).sum() / denom

        sl = y.rolling(lookback).apply(slope, raw=True)

        if normalize:
            sl = sl / y.rolling(lookback).mean().replace(0, np.nan)

        return sign_with_deadband(sl.fillna(0.0), entry=entry, exit=exit)

class BBExpansionTrend(Strategy):
    name = "bb_expansion_trend"

    def signal(self, df, width_lookback=20, entry_width=0.0, allow_flip=True):
        dwidth = df["bb_width"] - df["bb_width"].shift(width_lookback)
        expanding = dwidth > entry_width

        long_entry = expanding & (df["close"] > df["bb_upper"])
        short_entry = expanding & (df["close"] < df["bb_lower"])

        long_exit = df["close"] < df["bb_mid"]
        short_exit = df["close"] > df["bb_mid"]

        return stateful_signal_from_entries_exits(
            long_entry, long_exit, short_entry, short_exit, allow_flip=allow_flip
        )

class RSIMR(Strategy):
    name = "rsi_mr"

    def signal(self, df, lo=30, hi=70, exit_long=50, exit_short=50, allow_flip=True):
        rsi = df["rsi"]

        long_entry = rsi <= lo
        short_entry = rsi >= hi

        long_exit = rsi >= exit_long
        short_exit = rsi <= exit_short

        return stateful_signal_from_entries_exits(
            long_entry, long_exit, short_entry, short_exit, allow_flip=allow_flip
        )

class BollingerMR(Strategy):
    name = "bb_mr"

    def signal(self, df, exit_at="mid", allow_flip=True):
        long_entry = df["close"] < df["bb_lower"]
        short_entry = df["close"] > df["bb_upper"]

        if exit_at == "mid":
            long_exit = df["close"] >= df["bb_mid"]
            short_exit = df["close"] <= df["bb_mid"]
        elif exit_at == "inside":
            long_exit = df["close"] >= df["bb_lower"]
            short_exit = df["close"] <= df["bb_upper"]
        else:
            raise ValueError("exit_at must be 'mid' or 'inside'")

        return stateful_signal_from_entries_exits(
            long_entry, long_exit, short_entry, short_exit, allow_flip=allow_flip
        )

class PercentBMR(Strategy):
    name = "percentb_mr"

    def signal(self, df, lo=0.05, hi=0.95, exit_long=0.5, exit_short=0.5, allow_flip=True):
        pb = df["bb_percent_b"]

        long_entry = pb <= lo
        short_entry = pb >= hi

        long_exit = pb >= exit_long
        short_exit = pb <= exit_short

        return stateful_signal_from_entries_exits(
            long_entry, long_exit, short_entry, short_exit, allow_flip=allow_flip
        )

class KeltnerMR(Strategy):
    name = "kc_mr"

    def signal(self, df, exit_at="mid", allow_flip=True):
        long_entry = df["close"] < df["kc_lower"]
        short_entry = df["close"] > df["kc_upper"]

        if exit_at == "mid":
            long_exit = df["close"] >= df["kc_mid"]
            short_exit = df["close"] <= df["kc_mid"]
        elif exit_at == "inside":
            long_exit = df["close"] >= df["kc_lower"]
            short_exit = df["close"] <= df["kc_upper"]
        else:
            raise ValueError("exit_at must be 'mid' or 'inside'")

        return stateful_signal_from_entries_exits(
            long_entry, long_exit, short_entry, short_exit, allow_flip=allow_flip
        )

class StochRSIMR(Strategy):
    name = "stochrsi_mr"

    def signal(self, df, lo=20, hi=80, exit_long=50, exit_short=50, use_100=True, allow_flip=True):
        k = df["stoch_rsi_k_100"] if use_100 else (df["stoch_rsi_k"] * 100)

        long_entry = k <= lo
        short_entry = k >= hi

        long_exit = k >= exit_long
        short_exit = k <= exit_short

        return stateful_signal_from_entries_exits(
            long_entry, long_exit, short_entry, short_exit, allow_flip=allow_flip
        )

class VWAPFade(Strategy):
    name = "vwap_fade"

    def signal(self, df, entry_z=1.0, exit_z=0.25, allow_flip=True):
        if "atr" not in df:
            raise ValueError("VWAPFade requires 'atr' column")

        dev = (df["close"] - df["vwap"]) / df["atr"].replace(0, np.nan)

        long_entry = dev <= -entry_z
        short_entry = dev >= entry_z

        long_exit = dev >= -exit_z
        short_exit = dev <= exit_z

        return stateful_signal_from_entries_exits(
            long_entry, long_exit, short_entry, short_exit, allow_flip=allow_flip
        )

class ZScoreMR(Strategy):
    name = "zscore_mr"

    def signal(self, df, lookback=50, entry=2.0, exit=0.5, allow_flip=True):
        m = df["close"].rolling(lookback).mean()
        s = df["close"].rolling(lookback).std(ddof=0)
        z = (df["close"] - m) / s.replace(0, np.nan)

        long_entry = z <= -entry
        short_entry = z >= entry
        long_exit = z >= -exit
        short_exit = z <= exit

        return stateful_signal_from_entries_exits(
            long_entry, long_exit, short_entry, short_exit, allow_flip=allow_flip
        )

In [68]:
def apply_stop_loss_close(sig_raw: pd.Series, close: pd.Series, stop_pct=0.02) -> pd.Series:
    """
    sig_raw: target signal in {-1,0,1}
    close: close price series
    return: stopped signal in {-1,0,1}
    """
    sig_raw = sig_raw.fillna(0).astype(float).copy()
    close = close.astype(float)

    out = pd.Series(0.0, index=sig_raw.index)

    state = 0
    entry_price = np.nan

    for i in range(len(sig_raw)):
        desired = sig_raw.iat[i]
        px = close.iat[i]

        if state == 0:
            if desired == 1:
                state = 1
                entry_price = px
            elif desired == -1:
                state = -1
                entry_price = px

        elif state == 1:
            stop_hit = px <= entry_price * (1 - stop_pct)

            if stop_hit:
                state = 0
                entry_price = np.nan
            elif desired != 1:
                state = 0
                entry_price = np.nan

        elif state == -1:
            stop_hit = px >= entry_price * (1 + stop_pct)

            if stop_hit:
                state = 0
                entry_price = np.nan
            elif desired != -1:
                state = 0
                entry_price = np.nan

        out.iat[i] = state

    return out

## Backtest

In [70]:
cfg_1m = BTConfig(
    fee_bps=1.0,
    slippage_bps=1.0,
    max_leverage=3.0,
    vol_target=None,
    bars_per_year=365 * 24 * 60,
    embargo_bars=0,
    allow_short=True,
)

param_grid = [
    {"exit_at": "mid", "allow_flip": True},
    {"exit_at": "inside", "allow_flip": True},
]

In [81]:
regime_cols = [
    "atr_vol_regime_4h",
    "imom_label_4h",
    "imacd_color_p_4h",
]

In [82]:
grouping_dict = {
    "imom_imom": ["imom_label_4h", "imacd_color_p_4h"],
    "imom_atr_imom": ["imom_label_4h", "atr_vol_regime_4h", "imacd_color_p_4h"],
}

allowed_values = {
    "atr_vol_regime_4h": ["high_vol", "low_vol"],
    "imom_label_4h": ["plus","minus","flat"],
    "imacd_color_p_4h": ["lime", "green", "red", "orange"],
}

In [83]:
df_signal_1h.columns

Index(['t', 'open', 'high', 'low', 'close', 'vol_base', 'tr', 'atr', 'atr_pct',
       'hlc3', 'imacd_hi', 'imacd_lo', 'imacd_mi', 'imacd_md', 'imacd_sb',
       'imacd_sh', 'imacd_color', 'imacd_sb_atr', 'imacd_sh_atr',
       'atr_vol_regime', 'atr_vol_regime_enter_high',
       'atr_vol_regime_exit_high', 'imom_label', 'available_at_4h',
       'source_t_4h', 'atr_vol_regime_4h', 'imom_label_4h', 'imacd_color_4h',
       'imacd_color_p_4h', 'imacd_sb_atr_4h', 'imacd_sh_atr_4h', 'atr_pct_4h',
       'available_at_1d', 'source_t_1d', 'atr_vol_regime_1d', 'imom_label_1d',
       'imacd_color_1d', 'imacd_color_p_1d', 'imacd_sb_atr_1d',
       'imacd_sh_atr_1d', 'atr_pct_1d'],
      dtype='str')

In [84]:
summary_df, _ = scan_groupings_fast(
    df_exec=raw_1m,                           # 1m exec df
    df_signal=df_signal_1h,
    signal_tf="1h",
    strat=KeltnerMR(),
    param_grid=param_grid,
    cfg=cfg_1m,
    grouping_dict=grouping_dict,
    allowed_values=allowed_values,
    stop_pct=0.02,
    force_flat_outside=True,
    exec_delay_bars=1,
    strategy_name="KeltnerMR",
    extra_meta={"regime_tf": "4h", "signal_tf": "1h", "exec_tf": "1m"},
    keep_signal_cols=get_keep_signal_cols_for_strategy(
        strat=KeltnerMR(),
        df_signal=df_signal_1h,
        allowed_values=allowed_values,
    ),
    store_bt=False,   # เร็วสุด
    verbose=True,
)

KeyError: 'kc_lower'

# Part 4: Execution / position / fee / slippage

# Part 5: Metrics + scan